# Age-from-gait classifier feasibility

This is a parallel research direction, not an input to the primary stroke classifier. The first experiment uses only healthy Voisard participants because they are the only primary participants with age metadata and this avoids making age a proxy for stroke.

The target is an exploratory three-class age grouping: 18--39, 40--59 and 60+. Participant-level folds are used, and the source/class imbalance is handled with participant-level weighting.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'; INTERIM = PROJECT_ROOT / 'data' / 'interim'
MAG_PATH = PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'; MANIFEST_PATH = INTERIM / 'ml_readiness_manifest.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); torch.set_num_threads(4)
EPOCHS = 4; BATCH_SIZE = 128; AGE_GROUPS = ['young_18_39', 'middle_40_59', 'older_60_plus']; AGE_TO_INT = {g:i for i,g in enumerate(AGE_GROUPS)}
metadata = pd.read_csv(METADATA_PATH); manifest = pd.read_csv(MANIFEST_PATH); magnitude_windows = np.load(MAG_PATH, mmap_mode='r')
participant_age = manifest[manifest.dataset_id.eq('voisard_2025')].groupby('subject').age.first().astype(float).to_dict()
def age_group(age): return 'young_18_39' if age < 40 else ('middle_40_59' if age < 60 else 'older_60_plus')
participant_table = metadata[metadata.dataset_id.eq('voisard_2025') & metadata.label.eq('healthy')][['participant_key']].drop_duplicates()
participant_table['subject'] = participant_table.participant_key.str.split(':').str[-1]
participant_table['age'] = participant_table.subject.map(participant_age); participant_table['age_group'] = participant_table.age.map(age_group); participant_table['target'] = participant_table.age_group.map(AGE_TO_INT)
window_meta = metadata[metadata.participant_key.isin(participant_table.participant_key)].copy(); window_meta['target'] = window_meta.participant_key.map(participant_table.set_index('participant_key').target)
print('Device:', DEVICE, '| healthy participants:', len(participant_table), '| windows:', len(window_meta))
print(participant_table.groupby('age_group').size())

Device: cuda | healthy participants: 72 | windows: 1039
age_group
middle_40_59     15
older_60_plus    15
young_18_39      42
dtype: int64


In [2]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def pooled_stats(indices):
    total = np.zeros(3, dtype='float64'); total_sq = np.zeros(3, dtype='float64'); count = 0
    for start in range(0, len(indices), 512):
        batch = np.asarray(magnitude_windows[indices[start:start + 512]], dtype='float32')
        total += batch.sum(axis=(0, 1)); total_sq += np.square(batch).sum(axis=(0, 1)); count += batch.shape[0] * batch.shape[1]
    mean = total / count; std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-8)); return mean.astype('float32'), std.astype('float32')

def participant_weight(indices):
    frame = window_meta.loc[indices]; window_counts = frame.groupby('participant_key').size(); class_counts = frame.groupby('target').participant_key.nunique()
    w = frame.participant_key.map(1.0 / window_counts).to_numpy(); w *= frame.target.map(1.0 / class_counts).to_numpy(); return (w / w.mean()).astype('float32')

class AgeDataset(Dataset):
    def __init__(self, indices, mean, std, weights=None):
        self.indices = np.asarray(indices, dtype='int64'); self.mean = mean.reshape(1, 3); self.std = std.reshape(1, 3); self.weights = np.ones(len(self.indices), dtype='float32') if weights is None else weights
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item]); signal = ((np.asarray(magnitude_windows[index], dtype='float32') - self.mean) / self.std).T.copy()
        target = int(window_meta.loc[index, 'target'])
        return torch.from_numpy(signal), torch.tensor(target), torch.tensor(float(self.weights[item])), torch.tensor(index)

In [3]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=16):
        super().__init__(); bottleneck = min(32, in_channels); self.bottleneck = nn.Conv1d(in_channels, bottleneck, 1, bias=False)
        self.branches = nn.ModuleList([nn.Conv1d(bottleneck, out_channels, 7, padding=3, bias=False), nn.Conv1d(bottleneck, out_channels, 15, padding=7, bias=False), nn.Conv1d(bottleneck, out_channels, 25, padding=12, bias=False)])
        self.pool_branch = nn.Conv1d(in_channels, out_channels, 1, bias=False); self.bn = nn.BatchNorm1d(out_channels * 4); self.residual = nn.Conv1d(in_channels, out_channels * 4, 1, bias=False) if in_channels != out_channels * 4 else nn.Identity()
    def forward(self, x):
        z = self.bottleneck(x); branches = [branch(z) for branch in self.branches]; branches.append(self.pool_branch(nn.functional.max_pool1d(x, 3, stride=1, padding=1)))
        return nn.functional.gelu(self.bn(torch.cat(branches, dim=1)) + self.residual(x))

class AgeCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.features = nn.Sequential(InceptionBlock(3), nn.MaxPool1d(2), InceptionBlock(64), nn.AdaptiveAvgPool1d(1)); self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(64, 3))
    def forward(self, x): return self.classifier(self.features(x))

def predict(model, loader):
    model.eval(); rows = []
    with torch.no_grad():
        for signals, _, _, indices in loader:
            probs = torch.softmax(model(signals.to(DEVICE)), dim=1).cpu().numpy(); rows.extend(zip(indices.numpy(), probs))
    return {int(i): p for i, p in rows}

def participant_predictions(indices, window_probs, fold, seed):
    frame = window_meta.loc[indices, ['participant_key', 'target']].copy(); frame['probs'] = [window_probs[int(i)] for i in indices]
    grouped = frame.groupby(['participant_key', 'target'])['probs'].apply(lambda s: np.mean(np.stack(s.to_numpy()), axis=0)).reset_index()
    probs = np.stack(grouped.probs.to_numpy()); grouped['pred'] = probs.argmax(axis=1); grouped['p0'] = probs[:,0]; grouped['p1'] = probs[:,1]; grouped['p2'] = probs[:,2]; grouped['fold'] = fold; grouped['seed'] = seed
    return grouped

In [4]:
def train_one(train_participants, test_participants, fold, seed):
    set_seed(seed); train_keys = set(train_participants.participant_key); test_keys = set(test_participants.participant_key)
    train_indices = window_meta.index[window_meta.participant_key.isin(train_keys)].to_numpy(); test_indices = window_meta.index[window_meta.participant_key.isin(test_keys)].to_numpy()
    mean, std = pooled_stats(train_indices); weights = participant_weight(train_indices)
    train_loader = DataLoader(AgeDataset(train_indices, mean, std, weights), batch_size=BATCH_SIZE, shuffle=True, num_workers=0); test_loader = DataLoader(AgeDataset(test_indices, mean, std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    class_counts = train_participants.target.value_counts().sort_index().to_numpy(); class_weights = torch.tensor(len(train_participants) / (3 * class_counts), dtype=torch.float32, device=DEVICE)
    model = AgeCNN().to(DEVICE); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(EPOCHS):
        model.train()
        for signals, targets, weights_batch, _ in train_loader:
            optimizer.zero_grad(); logits = model(signals.to(DEVICE)); loss = (nn.functional.cross_entropy(logits, targets.to(DEVICE), weight=class_weights, reduction='none') * weights_batch.to(DEVICE)).mean(); loss.backward(); optimizer.step()
    predictions = participant_predictions(test_indices, predict(model, test_loader), fold, seed); y = predictions.target.to_numpy(); p = predictions[['p0','p1','p2']].to_numpy();
    return predictions, {'fold': fold, 'seed': seed, 'participants': len(predictions), 'balanced_accuracy': balanced_accuracy_score(y, predictions.pred), 'macro_f1': f1_score(y, predictions.pred, average='macro'), 'roc_auc_ovr': roc_auc_score(y, p, multi_class='ovr', average='macro')}

all_predictions = []; rows = []
for seed in [42, 52, 62]:
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (train_pos, test_pos) in enumerate(splitter.split(participant_table, participant_table.target)):
        pred, row = train_one(participant_table.iloc[train_pos], participant_table.iloc[test_pos], fold, seed); all_predictions.append(pred); rows.append(row); print('seed', seed, 'fold', fold, row)
predictions = pd.concat(all_predictions, ignore_index=True); results = pd.DataFrame(rows)
print('Mean results:'); print(results[['balanced_accuracy','macro_f1','roc_auc_ovr']].mean().round(3).to_string())
print('Confusion matrix on pooled OOF-like predictions by run is reported per fold; this is a feasibility study, not an external age benchmark.')

seed 42 fold 0 {'fold': 0, 'seed': 42, 'participants': 15, 'balanced_accuracy': 0.5555555555555555, 'macro_f1': 0.47222222222222215, 'roc_auc_ovr': 0.6728395061728395}


seed 42 fold 1 {'fold': 1, 'seed': 42, 'participants': 15, 'balanced_accuracy': 0.48148148148148145, 'macro_f1': 0.4678362573099415, 'roc_auc_ovr': 0.6296296296296297}


seed 42 fold 2 {'fold': 2, 'seed': 42, 'participants': 14, 'balanced_accuracy': 0.49999999999999994, 'macro_f1': 0.4666666666666666, 'roc_auc_ovr': 0.7222222222222223}


seed 42 fold 3 {'fold': 3, 'seed': 42, 'participants': 14, 'balanced_accuracy': 0.6388888888888888, 'macro_f1': 0.5166666666666667, 'roc_auc_ovr': 0.9286616161616162}


seed 42 fold 4 {'fold': 4, 'seed': 42, 'participants': 14, 'balanced_accuracy': 0.4166666666666667, 'macro_f1': 0.35555555555555557, 'roc_auc_ovr': 0.8169191919191919}


...

seed 62 fold 0 {'fold': 0, 'seed': 62, 'participants': 15, 'balanced_accuracy': 0.5555555555555555, 'macro_f1': 0.3148148148148148, 'roc_auc_ovr': 0.7685185185185185}


seed 62 fold 1 {'fold': 1, 'seed': 62, 'participants': 15, 'balanced_accuracy': 0.4444444444444444, 'macro_f1': 0.2692307692307692, 'roc_auc_ovr': 0.6450617283950617}


seed 62 fold 2 {'fold': 2, 'seed': 62, 'participants': 14, 'balanced_accuracy': 0.4444444444444444, 'macro_f1': 0.31746031746031744, 'roc_auc_ovr': 0.6622474747474747}


seed 62 fold 3 {'fold': 3, 'seed': 62, 'participants': 14, 'balanced_accuracy': 0.4444444444444444, 'macro_f1': 0.26666666666666666, 'roc_auc_ovr': 0.7058080808080809}


seed 62 fold 4 {'fold': 4, 'seed': 62, 'participants': 14, 'balanced_accuracy': 0.4444444444444444, 'macro_f1': 0.29304029304029305, 'roc_auc_ovr': 0.7563131313131312}
Mean results:
balanced_accuracy    0.451
macro_f1             0.342
roc_auc_ovr          0.726
Confusion matrix on pooled OOF-like predictions by run is reported per fold; this is a feasibility study, not an external age benchmark.


In [5]:
results.to_csv(PROCESSED / 'age_classifier_fold_results.csv', index=False); predictions.to_csv(PROCESSED / 'age_classifier_predictions.csv', index=False)
summary = results.agg({'balanced_accuracy':['mean','std'], 'macro_f1':['mean','std'], 'roc_auc_ovr':['mean','std']}).reset_index().rename(columns={'index':'statistic'})
summary.to_csv(PROCESSED / 'age_classifier_summary.csv', index=False)
print('Saved age classifier outputs.')

Saved age classifier outputs.


## Interpretation gate

A useful age model would support an aging-related gait analysis, not prove that age should be fed into the stroke model. The stroke model must still be reported by age-overlap strata and tested on age-matched external participants.